# Model Version 1

NIH 14 Chest X-ray downsized to 224 x 224. Swin_t is the model used. Atom optimized

In [1]:
import sys
print(sys.executable)

c:\Users\nick\AppData\Local\Programs\Python\Python311\python.exe


In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm


if __name__ == "__main__":

    # Load labels
    data_frame = pd.read_csv("../chest_xray_dataset/CXR8/Data_Entry_2017_v2020.csv")

    # Keep only the two columns we need
    data_frame = data_frame[["Image Index", "Finding Labels"]].copy()

    # Parse multi-label strings  e.g. "Atelectasis|Cardiomegaly"
    data_frame["labels"] = data_frame["Finding Labels"].str.split("|")

    ALL_CLASSES = [
        "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
        "Effusion", "Emphysema", "Fibrosis", "Hernia",
        "Infiltration", "Mass", "No Finding", "Nodule",
        "Pleural_Thickening", "Pneumonia", "Pneumothorax",
    ]
    NUM_CLASSES = len(ALL_CLASSES)

    mlb = MultiLabelBinarizer(classes=ALL_CLASSES)
    label_matrix = mlb.fit_transform(data_frame["labels"])  # (N, 15)

    # Build image-path index
    IMAGE_ROOT = r"C:\Users\nick\computing_for_health_and_medicine\chest_xray_dataset\CXR8\images"

    # Recursively find every PNG once and build a filename -> full-path dict
    all_png = glob.glob(os.path.join(IMAGE_ROOT, "**", "*.png"), recursive=True)
    path_lookup = {os.path.basename(p): p for p in all_png}
    print(f"Found {len(path_lookup):,} images on disk.")

    # Filter dataframe to images that actually exist
    mask = data_frame["Image Index"].isin(path_lookup)
    data_frame = data_frame[mask].reset_index(drop=True)
    label_matrix = label_matrix[mask.values]
    print(f"Matched {len(data_frame):,} rows after filtering.")

    # Train / val split
    indices = np.arange(len(data_frame))
    train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)

    # Dataset
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    val_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


    class CXR8Dataset(Dataset):
        def __init__(self, df, labels, idx_array, transform, lookup):
            self.df        = df.iloc[idx_array].reset_index(drop=True)
            self.labels    = labels[idx_array]
            self.transform = transform
            self.lookup    = lookup

        def __len__(self):
            return len(self.df)

        def __getitem__(self, i):
            fname = self.df.loc[i, "Image Index"]
            img   = Image.open(self.lookup[fname]).convert("RGB")
            img   = self.transform(img)
            lbl   = torch.tensor(self.labels[i], dtype=torch.float32)
            return img, lbl


    train_ds = CXR8Dataset(data_frame, label_matrix, train_idx, train_tf, path_lookup)
    val_ds   = CXR8Dataset(data_frame, label_matrix, val_idx,   val_tf,   path_lookup)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
    # Replace the classification head for multi-label output
    in_features = model.head.in_features
    model.head  = nn.Linear(in_features, NUM_CLASSES)
    model       = model.to(device)

    # Training setup
    criterion = nn.BCEWithLogitsLoss()          # multi-label
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

    # Train / eval loops
    def run_epoch(loader, train=True):
        model.train() if train else model.eval()
        total_loss = 0.0
        with torch.set_grad_enabled(train):
            for imgs, lbls in tqdm(loader, desc="train" if train else "val ", leave=False):
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = model(imgs)
                loss   = criterion(logits, lbls)
                if train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                total_loss += loss.item() * imgs.size(0)
        return total_loss / len(loader.dataset)


    NUM_EPOCHS = 10
    best_val   = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss  = run_epoch(train_loader, train=True)
        val_loss = run_epoch(val_loader,   train=False)
        scheduler.step()

        flag = ""
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "swin_cxr8_best.pth")
            flag = "  - saved"

        print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
            f"train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}{flag}")

    print("Done. Best val loss:", round(best_val, 4))

Found 112,120 images on disk.
Matched 112,120 rows after filtering.
Train: 95,302  |  Val: 16,818
Device: cuda


Epoch 01/10  train_loss=0.1951  val_loss=0.1885  ← saved


Epoch 02/10  train_loss=0.1852  val_loss=0.1824  ← saved


train:  76%|███████▌  | 2254/2979 [28:08<09:06,  1.33it/s]